# Which sellers should a limited budget go to first? (Phase 5)

Phase 4 found sellers whose on-time orders are unhappy more often than usual, beyond chance. Olist
can't help every seller at once. This notebook uses **linear programming** (PuLP) to choose which
sellers to include in a pilot under a fixed budget. The data has no costs, so every cost and benefit
is a written-down **assumption**. The output is a ranked list **worth testing**, not a promise of results.

**Contents:** 1. Setup · 2. A tiny example · 3. The seller table

## 1. Setup
**PuLP** lets us describe a choice problem in plain Python: what we can choose, what we want to
maximise, and the limits. A **solver** (here HiGHS, free and open source) then finds the best answer.

In [1]:
import pulp
print("PuLP", pulp.__version__, "| solvers available:", pulp.listSolvers(onlyAvailable=True))

PuLP 4.2.0 | solvers available: ['HiGHS']


## 2. A tiny example (made-up numbers)
Four sellers. Helping each one costs something and is expected to avoid some unhappy orders.
The budget is 10. Which sellers should we pick to avoid the most unhappy orders?

In [2]:
toy = {            # seller: (cost, unhappy orders avoided)  - made-up numbers for the example
    "A": (6, 30),
    "B": (5, 20),
    "C": (5, 20),
    "D": (3, 9),
}
BUDGET = 10

model = pulp.LpProblem("toy", pulp.LpMaximize)                        # we want the biggest benefit
pick = {s: model.add_variable(f"pick_{s}", cat="Binary") for s in toy}  # 1 = help this seller, 0 = don't
model += pulp.lpSum(toy[s][1] * pick[s] for s in toy)                 # goal: total unhappy orders avoided
model += pulp.lpSum(toy[s][0] * pick[s] for s in toy) <= BUDGET       # limit: total cost within budget
result = model.solve(pulp.HiGHS(msg=False))                           # the solver does the maths

chosen = [s for s in toy if pick[s].value() > 0.5]
print("status:", result.status.name, "| chosen:", chosen,
      "| cost:", sum(toy[s][0] for s in chosen), "| avoided:", sum(toy[s][1] for s in chosen))

status: Optimal | chosen: ['B', 'C'] | cost: 10 | avoided: 40


**Why not just take the "best" seller first?** Seller A has the biggest benefit, so a simple
"best first" rule takes A (cost 6), then only D still fits (cost 3): 39 avoided. The model finds
B + C (cost 10): 40 avoided. With a budget, the best *combination* is not always the best items
one by one, and that is the job PuLP does for us.

## 3. The seller table (real data)
The model needs one row per candidate seller. We rebuild the Phase 4 seller test exactly (reviewed,
**on-time**, single-seller, delivered orders; 50+ orders; one-sided binomial test; Holm), then keep
the sellers that are worse than usual beyond chance. For each one we add:
- **excess unhappy orders:** how many more unhappy orders they had than the usual rate would give
  (`unhappy - usual rate × orders`). This is the most a fix could hope to remove.
- **careful excess:** the same, using the low end of the seller's 95% Wilson interval, so a seller
  whose high rate may partly be luck counts for less.
- **months active and orders per month,** so later steps can talk about a pilot lasting a few months.

In [3]:
import sys
from pathlib import Path

import pandas as pd
from scipy import stats
from sqlalchemy import text

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))            # so "from src..." works when run from notebooks/
from src.db import get_engine

engine = get_engine()

def query(sql: str) -> pd.DataFrame:
    """Run one SQL query and return the result as a table."""
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn)

def holm(p_values: list) -> list:
    """Holm correction for many tests at once (same as notebook 02)."""
    m, order = len(p_values), sorted(range(len(p_values)), key=lambda i: p_values[i])
    adjusted, running = [0.0] * m, 0.0
    for rank, i in enumerate(order):
        running = max(running, min(1.0, (m - rank) * p_values[i]))
        adjusted[i] = running
    return adjusted

In [4]:
sellers = query("""
    SELECT f.seller_id, s.state,
           count(*)                             AS orders,     -- reviewed on-time orders
           count(*) FILTER (WHERE f.is_unhappy) AS unhappy,
           min(f.purchase_date) AS first_order, max(f.purchase_date) AS last_order
    FROM analytics.fact_order f
    JOIN analytics.dim_seller s ON s.seller_id = f.seller_id
    WHERE f.n_sellers = 1 AND f.has_review AND f.order_status = 'delivered' AND f.days_late <= 0
    GROUP BY f.seller_id, s.state
""")
usual = sellers["unhappy"].sum() / sellers["orders"].sum()        # usual on-time unhappy rate

tested = sellers[sellers["orders"] >= 50].copy()                  # same 50+ rule as Phase 4
tested["p_value"] = [stats.binomtest(int(u), int(n), usual, alternative="greater").pvalue
                     for u, n in zip(tested["unhappy"], tested["orders"])]
tested["p_holm"] = holm(list(tested["p_value"]))
print(f"usual rate {100 * usual:.2f}% | {len(tested)} sellers tested | "
      f"{(tested['p_holm'] < 0.05).sum()} worse than usual after Holm")

usual rate 8.77% | 393 sellers tested | 24 worse than usual after Holm


In [5]:
cand = tested[tested["p_holm"] < 0.05].copy()                      # the candidates for the pilot
cand["rate_pct"] = 100 * cand["unhappy"] / cand["orders"]
cand["rate_low_pct"] = [100 * stats.binomtest(int(u), int(n)).proportion_ci(method="wilson").low
                        for u, n in zip(cand["unhappy"], cand["orders"])]
cand["excess"] = cand["unhappy"] - usual * cand["orders"]                       # unhappy orders above usual
cand["excess_careful"] = (cand["rate_low_pct"] / 100 - usual) * cand["orders"]  # same, at the interval's low end
days = (pd.to_datetime(cand["last_order"]) - pd.to_datetime(cand["first_order"])).dt.days
cand["months_active"] = (days / 30.44).clip(lower=1)                            # at least 1 month
cand["orders_per_month"] = cand["orders"] / cand["months_active"]
cand["seller"] = cand["seller_id"].str[:8]                                      # short id for display

cand = cand.sort_values("excess", ascending=False).reset_index(drop=True)
cols = ["seller", "state", "orders", "unhappy", "rate_pct", "rate_low_pct",
        "excess", "excess_careful", "months_active", "orders_per_month"]
cand[cols].round(1)

,seller,state,orders,unhappy,rate_pct,rate_low_pct,excess,excess_careful,months_active,orders_per_month
0,7c67e144,SP,870,178,20.5,17.9,101.7,79.5,18.3,47.5
1,6560211a,SP,1680,221,13.2,11.6,73.7,47.9,18.3,91.6
2,2eb70248,SP,162,74,45.7,38.2,59.8,47.7,5.4,30.3
3,4a3ca931,SP,1489,178,12.0,10.4,47.4,24.4,19.6,76.0
4,1ca7077d,SP,87,47,54.0,43.6,39.4,30.3,6.8,12.9
5,d2374cbc,SP,458,74,16.2,13.1,33.8,19.7,18.3,25.0
6,cca3071e,SP,624,87,13.9,11.4,32.3,16.7,22.6,27.6
7,710e3548,PR,124,37,29.8,22.5,26.1,17.0,22.3,5.6
8,897060da,SP,266,49,18.4,14.2,25.7,14.5,16.0,16.7
9,7040e82f,SP,185,40,21.6,16.3,23.8,13.9,11.5,16.1


In [6]:
total, top5 = cand["excess"].sum(), cand["excess"].head(5).sum()
print(f"{len(cand)} candidate sellers | {cand['orders'].sum():,} on-time orders | "
      f"{total:.0f} excess unhappy orders ({cand['excess_careful'].sum():.0f} careful)")
print(f"the 5 biggest hold {100 * top5 / total:.0f}% of the excess | "
      f"orders per seller: {cand['orders'].min()} to {cand['orders'].max():,}")

24 candidate sellers | 7,821 on-time orders | 686 excess unhappy orders (425 careful)
the 5 biggest hold 47% of the excess | orders per seller: 57 to 1,680


**What the table shows.** The rebuild matches Phase 4 (usual rate 8.77%, 393 sellers tested, 24
worse than usual). Together the 24 sellers had 686 more unhappy on-time orders than the usual rate
would give (425 using the careful version), and the 5 biggest hold 47% of that excess.
- **Rate and excess rank sellers differently.** The highest rate (54.0%, 87 orders) is only 5th by
  excess, while a seller at 13.2% is 2nd because it has 1,680 orders.
- **The careful column shrinks small sellers most,** because their intervals are wider.
- **Most candidates are in SP** (São Paulo state), like most Olist sellers.

These excess counts are the most a fix *could* remove, over about 1.5 years of orders. How much a
real fix would remove, and what it would cost, are assumptions set in the next step.